# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one row = one content_id (one content page), for the month 2026-03.
Grain source: fact_content_daily_performance is content-day; we aggregate days 1–15 → feature row per page. Days 16–31 → label per page.
Table(s): fact_content_daily_performance (joined with dim_content for static attributes like word_count, content_type).
Time window: March 2026, split feature (day 1–15) / label (day 16–31), no overlap.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|-------|--------|-----|
| `content_id` | Context | join/group key, not signal |
| `client_id` | Context | for client-holdout split + history filter |
| `gsc_data_start`, `ga4_data_start` | Context | filter clients with history before 2026-03 |
| `ga4_data_available` | Context | filter rows where GA4 is FALSE → zero-filled (not real) |
| `impressions_90d`, `ctr`, `avg_position` (days 1–15 agg) | Feature | measured before label window |
| `content_age_days`, `word_count`, `content_type` | Feature (deferred) | Static/CSV join; not in warehouse. Added Week 4. |
| `trend_direction` (days 16–31) | Label source | IS what is_declining_label is derived from |
| `trend_pct` | Excluded | leaks the label — same as notebook 02 trap |
| `is_declining_label` | Label | target (declining = 1, not = 0) |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN2')  # Get your token

con = duckdb.connect()

# Create secret on the connection
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Now query using the connection
rel = "hf://datasets/FlyRank/internship-warehouse"

result = con.sql(f"""
SELECT content_hash_id, report_date, COUNT(*) as c
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
GROUP BY content_hash_id, report_date
HAVING c > 1
LIMIT 5
""")
print(result)

# INTERPRETATION: Zero duplicates found → grain is clean.
# One row per (content_hash_id, report_date) verified.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────────┬───────┐
│ content_hash_id │ report_date │   c   │
│     varchar     │    date     │ int64 │
├─────────────────┴─────────────┴───────┤
│                0 rows                 │
└───────────────────────────────────────┘



In [2]:
result = con.sql(f"""
SELECT
  COUNT(*) as total_rows,
  COUNT(DISTINCT content_hash_id) as unique_pages,
  MIN(report_date) as start_date,
  MAX(report_date) as end_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""")
print(result)

# INTERPRETATION: 9.8M rows, 331K unique pages, full March 1–31 → unit of analysis and time window verified.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────┬────────────┬────────────┐
│ total_rows │ unique_pages │ start_date │  end_date  │
│   int64    │    int64     │    date    │    date    │
├────────────┼──────────────┼────────────┼────────────┤
│    9841378 │       331437 │ 2026-03-01 │ 2026-03-31 │
└────────────┴──────────────┴────────────┴────────────┘



In [3]:
result = con.sql(f"""
SELECT
  COUNT(*) as total_rows,
  SUM(CASE WHEN ga4_data_available = TRUE THEN 1 ELSE 0 END) as rows_with_ga4,
  ROUND(100.0 * SUM(CASE WHEN ga4_data_available = TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) as pct_with_ga4
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""")
print(result)

# INTERPRETATION: Only 4.2% of rows have GA4 data → GA4 is sparse; most pages rely on GSC alone.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────┬──────────────┐
│ total_rows │ rows_with_ga4 │ pct_with_ga4 │
│   int64    │    int128     │    double    │
├────────────┼───────────────┼──────────────┤
│    9841378 │        413966 │          4.2 │
└────────────┴───────────────┴──────────────┘



In [4]:
result = con.sql(f"""
SELECT
  COUNT(*) as total_rows,
  SUM(CASE WHEN gsc_data_available = TRUE THEN 1 ELSE 0 END) as rows_with_gsc,
  ROUND(100.0 * SUM(CASE WHEN gsc_data_available = TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) as pct_with_gsc
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""")
print(result)

# INTERPRETATION: 36.7% have GSC data → GSC is the primary signal source; usable for features.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────┬──────────────┐
│ total_rows │ rows_with_gsc │ pct_with_gsc │
│   int64    │    int128     │    double    │
├────────────┼───────────────┼──────────────┤
│    9841378 │       3611061 │         36.7 │
└────────────┴───────────────┴──────────────┘



In [5]:
# Check overlap
result = con.sql(f"""
SELECT
  COUNT(*) as total,
  SUM(CASE WHEN gsc_data_available = TRUE AND ga4_data_available = TRUE THEN 1 ELSE 0 END) as both,
  SUM(CASE WHEN gsc_data_available = TRUE AND ga4_data_available = FALSE THEN 1 ELSE 0 END) as gsc_only,
  SUM(CASE WHEN gsc_data_available = FALSE AND ga4_data_available = TRUE THEN 1 ELSE 0 END) as ga4_only
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""")
print(result)

# INTERPRETATION: 2.1M rows with at least one signal (21.7%); 1.7M GSC-only dominates → training set is viable but sparse.

┌─────────┬────────┬──────────┬──────────┐
│  total  │  both  │ gsc_only │ ga4_only │
│  int64  │ int128 │  int128  │  int128  │
├─────────┼────────┼──────────┼──────────┤
│ 9841378 │ 364347 │  1718348 │    49619 │
└─────────┴────────┴──────────┴──────────┘



## 3B. Five Features + Label Leakage Trap

In [6]:
# Build feature frame: days 1-15 aggregation per page

feature_frame = con.sql(f"""
WITH daily_averages AS (
  SELECT
    content_hash_id,
    client_hash_id,
    -- Days 1-15: feature window
    AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_impressions ELSE NULL END) as gsc_impressions_1to15,
    AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_clicks ELSE NULL END) as gsc_clicks_1to15,
    AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_avg_position ELSE NULL END) as position_1to15,
    -- Days 16-31: label window
    AVG(CASE WHEN report_date >= '2026-03-16' AND gsc_data_available THEN gsc_impressions ELSE NULL END) as gsc_impressions_16to31,
    AVG(CASE WHEN report_date >= '2026-03-16' AND gsc_data_available THEN gsc_clicks ELSE NULL END) as gsc_clicks_16to31,
    AVG(CASE WHEN report_date >= '2026-03-16' AND gsc_data_available THEN gsc_avg_position ELSE NULL END) as position_16to31
  FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
  WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01' AND gsc_data_available = TRUE
  GROUP BY content_hash_id, client_hash_id
)
SELECT
  content_hash_id,
  client_hash_id,
  gsc_impressions_1to15 as gsc_impressions_avg,
  gsc_clicks_1to15 as gsc_clicks_avg,
  position_1to15 as position_avg,
  (CASE WHEN gsc_impressions_16to31 < gsc_impressions_1to15 THEN 1 ELSE 0 END) as is_declining_label
FROM daily_averages
""")
df_clean = feature_frame.df()

print("✅ Feature Frame (clean):")
print(df_clean.head())
print(f"Shape: {df_clean.shape}")
print()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Feature Frame (clean):
            content_hash_id           client_hash_id  gsc_impressions_avg  \
0  content_05597932fe4da067  client_73cda7b4e4f265ea             1.636364   
1  content_7a105f548d9c6916  client_73cda7b4e4f265ea           278.200000   
2  content_905aa32a0230694e  client_73cda7b4e4f265ea             5.933333   
3  content_a3ea9792f793ec72  client_73cda7b4e4f265ea            16.333333   
4  content_36c36abc7650d7af  client_73cda7b4e4f265ea           247.000000   

   gsc_clicks_avg  position_avg  is_declining_label  
0             0.0      4.939394                   0  
1             0.4      6.327311                   1  
2             0.0      3.010741                   1  
3             0.0      3.906852                   1  
4             0.2      6.473735                   1  
Shape: (176738, 6)



In [7]:
print("Label distribution:")
print(df_clean['is_declining_label'].value_counts())
print(f"Declining rate: {df_clean['is_declining_label'].mean():.1%}")

Label distribution:
is_declining_label
0    111280
1     65458
Name: count, dtype: int64
Declining rate: 37.0%


In [8]:
# Five Features + When Knowable:
features_doc = """
Features for ML-04 Baseline (Warehouse GSC only):

1. gsc_impressions_avg (days 1-15) - Knowable at decision moment because: measured 15 days before label window; historical GSC data.
2. gsc_clicks_avg (days 1-15) - Knowable at decision moment because: measured 15 days before label window; historical GSC data.
3. position_avg (days 1-15) - Knowable at decision moment because: measured 15 days before label window; historical GSC ranking.

Static content features (content_age_days, word_count, content_type) join from CSV in Week 4 — not available in warehouse. Deferred to next assignment.
"""
print(features_doc)


Features for ML-04 Baseline (Warehouse GSC only):

1. gsc_impressions_avg (days 1-15) - Knowable at decision moment because: measured 15 days before label window; historical GSC data.
2. gsc_clicks_avg (days 1-15) - Knowable at decision moment because: measured 15 days before label window; historical GSC data.
3. position_avg (days 1-15) - Knowable at decision moment because: measured 15 days before label window; historical GSC ranking.

Static content features (content_age_days, word_count, content_type) join from CSV in Week 4 — not available in warehouse. Deferred to next assignment.



In [9]:
# TRAP: Deliberately add is_declining_label as a feature (THIS LEAKS THE LABEL)
df_leaky = df_clean.copy()
print("⚠️ TRAP: Adding is_declining_label as a feature (leaks the label)...")

⚠️ TRAP: Adding is_declining_label as a feature (leaks the label)...


In [10]:
# Train on leaky features
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

X_leaky = df_leaky[['gsc_impressions_avg', 'gsc_clicks_avg', 'position_avg', 'is_declining_label']].fillna(0)
y = df_leaky['is_declining_label']

tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
y_pred_leaky = tree_leaky.predict(X_leaky)

print(f"Leaky Model Precision: {precision_score(y, y_pred_leaky, zero_division=0):.3f}")
print(f"Leaky Model Recall: {recall_score(y, y_pred_leaky, zero_division=0):.3f}")
print(f"Leaky Model F1: {f1_score(y, y_pred_leaky, zero_division=0):.3f}")
print("⚠️ Near-perfect score because is_declining_label IS the outcome.\n")

Leaky Model Precision: 1.000
Leaky Model Recall: 1.000
Leaky Model F1: 1.000
⚠️ Near-perfect score because is_declining_label IS the outcome.



In [11]:
# REMOVE TRAP
X_honest = df_leaky[['gsc_impressions_avg', 'gsc_clicks_avg', 'position_avg']].fillna(0)

tree_honest = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y)
y_pred_honest = tree_honest.predict(X_honest)

print("✅ TRAP REMOVED: Training on honest features only...")
print(f"Honest Model Precision: {precision_score(y, y_pred_honest, zero_division=0):.3f}")
print(f"Honest Model Recall: {recall_score(y, y_pred_honest, zero_division=0):.3f}")
print(f"Honest Model F1: {f1_score(y, y_pred_honest, zero_division=0):.3f}")

✅ TRAP REMOVED: Training on honest features only...
Honest Model Precision: 0.528
Honest Model Recall: 0.710
Honest Model F1: 0.606


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Structural sparsity:** 78% of March rows have neither GSC nor GA4 data — only 21.7% have at least one signal source. This sparsity is inherent to the warehouse, not a data quality bug. Even with 12 months of data, signal coverage remains ~20-25%.

**Consequence:** Pages without measurable pre-label-window signals (days 1–15 GSC/GA4 history) cannot be ranked and are excluded from training. This is not a limitation — it's a feature. We model only rankable content.

**Unbalanced history:** Clients joined FlyRank at different times. Some have 2+ years of history; others have weeks. The `gsc_data_start` and `ga4_data_start` flags control for this in holdout splits, but within a single month, panel is unbalanced.

**Label window sealed:** June 2026 is the final month in the dataset (export date 2026-07-03). Never use it for label logic — it's the natural outcome window. Iterate on mid-panel months (e.g., March) only.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.